# Install packages

In [ ]:
!pip install pymatgen

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.4/883.4 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 9.6 MB/s eta 0:00:00
  Created wheel for bibtexparser: filename=bibtexparser-1.4.4-py3-none-any.whl size=43609 sha256=626269df0836e369e7c581775a5d0478e0c7235c3b3c465052a02f8cae3200f1
  Stor

# Adding H to orthorhombic phas CuTA

In [ ]:
!pip install pymatgen -q

from pymatgen.core import Structure
from pymatgen.io.cif import CifWriter
from collections import Counter
import numpy as np

# ── 1. Load structure ──────────────────────────────────────────────────────────
s = Structure.from_file("/content/CuTA_orthorhombic.cif")

print("=" * 50)
print("ORIGINAL STRUCTURE")
print("=" * 50)
print(f"Formula    : {s.formula}")
print(f"Space group: {s.get_space_group_info()}")
print(f"Lattice    : a={s.lattice.a:.4f}, b={s.lattice.b:.4f}, c={s.lattice.c:.4f} Å")
print(f"Total sites: {len(s)}")

elem_count = Counter([str(site.specie) for site in s])
print(f"Element counts: {dict(elem_count)}")
print(f"\nExpecting {elem_count['C']} H atoms (1 per C, triazolate ring)")

# ── 2. Add 1 H per C atom ─────────────────────────────────────────────────────
# In triazolate (C2N3H2 ligand), every C has exactly 1 H.
# We place H by pointing AWAY from the average of its bonded neighbors.
# C-H bond length = 0.96 Å (standard aromatic)

print("\n" + "=" * 50)
print("ADDING HYDROGEN ATOMS (1 per C)")
print("=" * 50)

s_with_H = s.copy()
C_H_BOND = 0.96  # Å

# Build neighbor list manually using distance cutoff
# (avoids CrystalNN being too conservative)
C_N_CUTOFF = 1.6  # Å — max C-N bond
C_C_CUTOFF = 1.6  # Å — max C-C bond
SEARCH_CUTOFF = 2.5  # Å — search radius for neighbors

h_positions = []
c_indices = [i for i, site in enumerate(s_with_H) if str(site.specie) == 'C']

print(f"Found {len(c_indices)} C atoms in unit cell")

for i in c_indices:
    site = s_with_H[i]
    cart_pos = site.coords

    # Get all neighbors within cutoff
    neighbors = s_with_H.get_neighbors(site, SEARCH_CUTOFF)
    # Keep only C and N neighbors (exclude Cu and O)
    bond_neighbors = [n for n in neighbors if str(n.specie) in ('C', 'N')]

    if len(bond_neighbors) == 0:
        print(f"  WARNING: Site {i} C has no neighbors — skipping")
        continue

    # H points away from average bond direction
    bond_vectors = [cart_pos - n.coords for n in bond_neighbors]
    avg_vector = np.mean(bond_vectors, axis=0)
    norm = np.linalg.norm(avg_vector)

    if norm < 1e-6:
        print(f"  WARNING: Site {i} C has degenerate bonds — skipping")
        continue

    h_direction = avg_vector / norm
    h_cart = cart_pos + C_H_BOND * h_direction
    h_frac = (s_with_H.lattice.get_fractional_coords(h_cart)) % 1.0

    h_positions.append(h_frac)
    neighbor_labels = [str(n.specie) for n in bond_neighbors]
    print(f"  C site {i:>3}: neighbors={neighbor_labels} → H at {h_frac.round(4)}")

# Append all H atoms
for h_frac in h_positions:
    s_with_H.append("H", h_frac, validate_proximity=False)

print(f"\nH atoms added : {len(h_positions)}")
print(f"New formula   : {s_with_H.formula}")

# ── 3. Verify against cubic reference ─────────────────────────────────────────
print("\n" + "=" * 50)
print("VERIFICATION")
print("=" * 50)
new_counts = Counter([str(site.specie) for site in s_with_H])
print(f"{'Element':<10} {'Original':>10} {'With H':>10}")
print("-" * 32)
original_counts = Counter([str(site.specie) for site in s])
for el in sorted(set(list(original_counts) + list(new_counts))):
    print(f"{el:<10} {original_counts.get(el,0):>10} {new_counts.get(el,0):>10}")

c_count = new_counts.get('C', 0)
h_count = new_counts.get('H', 0)
print(f"\nC:H ratio = {c_count}:{h_count} (should be 1:1 like cubic reference)")
if c_count == h_count:
    print("✓ C:H ratio correct!")
else:
    print(f"✗ Mismatch — check neighbor search cutoff")

# ── 4. Export ──────────────────────────────────────────────────────────────────
out_cif = "/content/CuTA_orthorhombic_with_H.cif"
CifWriter(s_with_H).write_file(out_cif)
print(f"\nSaved to: {out_cif}")

ORIGINAL STRUCTURE
Formula    : Cu12 C48 N72 O4
Space group: ('Pnma', 62)
Lattice    : a=11.4404, b=11.7615, c=16.8950 Å
Total sites: 136
Element counts: {'Cu': 12, 'C': 48, 'N': 72, 'O': 4}

Expecting 48 H atoms (1 per C, triazolate ring)

ADDING HYDROGEN ATOMS (1 per C)
Found 48 C atoms in unit cell
  C site  12: neighbors=['N', 'C', 'N', 'N'] → H at [0.2089 0.75   0.11  ]
  C site  13: neighbors=['N', 'N', 'C', 'N'] → H at [0.7911 0.25   0.89  ]
  C site  14: neighbors=['C', 'N', 'N', 'N'] → H at [0.2911 0.25   0.61  ]
  C site  15: neighbors=['N', 'N', 'N', 'C'] → H at [0.7089 0.75   0.39  ]
  C site  16: neighbors=['N', 'N', 'C', 'N'] → H at [0.4172 0.75   0.0702]
  C site  17: neighbors=['N', 'N', 'N', 'C'] → H at [0.5828 0.25   0.9298]
  C site  18: neighbors=['N', 'N', 'N', 'C'] → H at [0.0828 0.25   0.5702]
  C site  19: neighbors=['N', 'N', 'C', 'N'] → H at [0.9172 0.75   0.4298]
  C site  20: neighbors=['N', 'N', 'N', 'C'] → H at [0.7913 0.5566 0.492 ]
  C site  21: neighbor

/tmp/ipykernel_900/1215661365.py:104: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Cu1', 'Cu1', 'Cu1', 'Cu1', 'Cu2', 'Cu2', 'Cu2', 'Cu2', 'Cu3', 'Cu3', 'Cu3', 'Cu3', 'C1', 'C1', 'C1', 'C1', 'C2', 'C2', 'C2', 'C2', 'C3', 'C3', 'C3', 'C3', 'C3', 'C3', 'C3', 'C3', 'C4', 'C4', 'C4', 'C4', 'C4', 'C4', 'C4', 'C4', 'C5', 'C5', 'C5', 'C5', 'C5', 'C5', 'C5', 'C5', 'C6', 'C6', 'C6', 'C6', 'C6', 'C6', 'C6', 'C6', 'C7', 'C7', 'C7', 'C7', 'C7', 'C7', 'C7', 'C7', 'N1', 'N1', 'N1', 'N1', 'N2', 'N2', 'N2', 'N2', 'N3', 'N3', 'N3', 'N3', 'N4', 'N4', 'N4', 'N4', 'N4', 'N4', 'N4', 'N4', 'N5', 'N5', 'N5', 'N5', 'N5', 'N5', 'N5', 'N5', 'N6', 'N6', 'N6', 'N6', 'N6', 'N6', 'N6', 'N6', 'N7', 'N7', 'N7', 'N7', 'N7', 'N7', 'N7', 'N7', 'N8', 'N8', 'N8', 'N8', 'N8', 'N8', 'N8', 'N8', 'N9', 'N9', 'N9', 'N9', 'N9', 'N9', 'N9', 'N9', 'N10', 'N10', 'N10', 'N10', 'N10', 'N10', 'N10', 'N10', 'N11',